<a href="https://colab.research.google.com/github/Gianbattistabsn/FAIML-RL-26/blob/alessandro-PPO-SAC/part2/clone_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

import os

REPO_URL = "https://github.com/Gianbattistabsn/FAIML-RL-26.git"
REPO_BRANCH = "dev"

REPO_ROOT = "/content/FAIML-RL-26"
VENV = "/content/rl_env"

# ------------------------------------------------------------
# Mount Drive
# ------------------------------------------------------------

# from google.colab import drive
# drive.mount('/content/drive')

# ------------------------------------------------------------
# Clone Repo
# ------------------------------------------------------------

if not os.path.exists(REPO_ROOT):
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}

# ------------------------------------------------------------
# Install venv support
# ------------------------------------------------------------

!apt-get update -qq
!apt-get install -y python3-venv

# ------------------------------------------------------------
# Recreate clean venv (IMPORTANT FIXED VERSION)
# ------------------------------------------------------------

if os.path.exists(VENV):
    !rm -rf {VENV}

!python -m venv --without-pip {VENV}

PYTHON = f"{VENV}/bin/python"

# install pip inside venv
!curl -sS https://bootstrap.pypa.io/get-pip.py | {PYTHON}

PIP = f"{VENV}/bin/pip"

# ------------------------------------------------------------
# Upgrade tooling
# ------------------------------------------------------------

!{PIP} install -U pip setuptools wheel

# ------------------------------------------------------------
# Install RL stack
# ------------------------------------------------------------

!{PIP} install \
    numpy==1.26.4 \
    gymnasium==0.29.1 \
    stable-baselines3==2.3.2 \
    pybullet \
    tensorboard \
    pyvirtualdisplay \
    moviepy \
    imageio \
    imageio-ffmpeg \
    shimmy \
    opencv-python-headless==4.9.0.80

# ------------------------------------------------------------
# Install LOCAL panda-gym
# ------------------------------------------------------------

!{PIP} install -e /content/FAIML-RL-26/part2/panda-gym

print("\nSETUP COMPLETE")
print("Python executable:", PYTHON)

Mounted at /content/drive
Cloning into '/content/FAIML-RL-26'...
remote: Enumerating objects: 512, done.
remote: Counting objects: 100% (195/195), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 512 (delta 113), reused 137 (delta 65), pack-reused 317 (from 1)
Receiving objects: 100% (512/512), 14.51 MiB | 33.85 MiB/s, done.
Resolving deltas: 100% (221/221), done.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  python3-pip-whl python3-setuptools-whl python3.10-venv
The following NEW packages will be installed:
  python3-pip-whl python3-setuptools-whl python3-venv python3.10-venv
0 upgraded, 4 newly installed, 0 to remove and 106 not upgraded.
Need to get 2,482 kB of archives.


In [2]:

PYTHON = "/content/rl_env/bin/python"

test_code = """
import numpy as np
import gymnasium
import pybullet
import panda_gym
import torch

print("NumPy:", np.__version__)
print("Gymnasium:", gymnasium.__version__)
print("Torch:", torch.__version__)
print("OK")
"""

with open("/content/test_env.py", "w") as f:
    f.write(test_code)

!{PYTHON} /content/test_env.py


pybullet build time: May  9 2026 22:18:02
NumPy: 1.26.4
Gymnasium: 0.29.1
Torch: 2.11.0+cu130
OK


In [3]:

PIP = "/content/rl_env/bin/pip"
REQ = "/content/FAIML-RL-26/requirements.txt"

!cat $REQ



gymnasium
mujoco
stable-baselines3[extra]>=2.0.0
imageio
opencv-python<4

In [4]:

! /content/rl_env/bin/pip install tqdm rich


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [rich]


In [5]:
!/content/rl_env/bin/pip install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 53.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 74.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 29.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [wandb]


In [6]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: gianbattista-busonera (gianbattista-busonera-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
## training
PYTHON = "/content/rl_env/bin/python"
SCRIPT = "/content/FAIML-RL-26/part2/train_sb3.py"

ENV_TYPE = "source"
SAMPLING = "none"
TIMESTEPS = 300_000
MASS_MIN = 0.5
MASS_MAX = 2
ADR_DELTA = 0.2
ADR_BUFFER_SIZE = 20
ADR_PERF_LOW = -25
ADR_PERF_HIGH = -10
ADR_BOUNDARY_PROB = 0.8

MASS_RANGE = f"{MASS_MIN} {MASS_MAX}"

!MPLBACKEND=Agg {PYTHON} {SCRIPT} \
    --env-type {ENV_TYPE} \
    --sampling-strategy {SAMPLING} \
    --timesteps {TIMESTEPS} \
    --no-vecnormalize \
    --mass-range {MASS_RANGE} \
    --adr-delta {ADR_DELTA} \
    --adr-buffer-size {ADR_BUFFER_SIZE} \
    --adr-perf-low {ADR_PERF_LOW} \
    --adr-perf-high {ADR_PERF_HIGH} \
    --adr-boundary-prob {ADR_BOUNDARY_PROB}

/content/FAIML-RL-26/part2/train_sb3.py:9: SyntaxWarning: invalid escape sequence '\p'
  python .\part2\train_sb3.py --env-type source --sampling-strategy none --timesteps 300000
use PPO? [y/n]
>object address  : 0x7bb5ccf417e0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


In [ ]:
# see performance on target
evaluate = False
if evaluate:
    PYTHON = "/content/rl_env/bin/python"
    SCRIPT = "/content/FAIML-RL-26/part2/eval_sb3.py"

    ENV_TYPE   = "target"
    MODEL_PATH = "/content/part2/models/ppo_push_udr_source_300k.zip"
    EPISODES   = 1_000
    STOCHASTIC = False
    USE_VECNORM = True

    extra = "" if USE_VECNORM else "--no-vecnormalize "
    extra += "--stochastic" if STOCHASTIC else ""

    !MPLBACKEND=Agg {PYTHON} {SCRIPT} \
        --model-path {MODEL_PATH} \
        --env-type {ENV_TYPE} \
        --episodes {EPISODES} {extra}